In [2]:
import torch
from sklearn.datasets import fetch_california_housing
housing = fetch_california_housing()

In [5]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import TensorDataset, DataLoader

X = housing['data']
y = housing['target']

X_train_full, X_test, y_train_full, y_test = train_test_split(X,y)
X_train, X_valid, y_train, y_valid = train_test_split(X_train_full,y_train_full)

print(X_train.shape, X_test.shape, X_valid.shape)

scl = StandardScaler()
scl.fit(X_train)

X_train = scl.transform(X_train)
X_test = scl.transform(X_test)
X_valid = scl.transform(X_valid)

X_train = torch.FloatTensor(X_train)
X_test = torch.FloatTensor(X_test)
X_valid = torch.FloatTensor(X_valid)

y_train = torch.FloatTensor(y_train).view(-1,1)
y_test = torch.FloatTensor(y_test).view(-1,1)
y_valid = torch.FloatTensor(y_valid).view(-1,1)

(11610, 8) (5160, 8) (3870, 8)


In [11]:
train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)
valid_dataset = TensorDataset(X_valid, y_valid)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)
valid_loader = DataLoader(valid_dataset, batch_size=32)

In [12]:
import torch.nn as nn
import torchmetrics
import matplotlib.pyplot as plt
import numpy as np

In [13]:
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

device

'cpu'

In [26]:
learning_rate = 0.01
model = nn.Sequential(
	nn.Linear(in_features=8, out_features=30), 
	nn.Sigmoid(),
	nn.Linear(in_features=30, out_features=50), 
	nn.Sigmoid(),
	nn.Linear(in_features=50, out_features=1)
)
criterion = nn.MSELoss()
optimizer = torch.optim.SGD(params=model.parameters(), lr=learning_rate)
metric = torchmetrics.MeanAbsoluteError().to(device)

history = {
	'loss' : [],
	'train_metric' : [],
}
n_epochs = 10

for epoch in range(n_epochs):
	total_loss = 0
	metric.reset()
	model.train()
	for X_batch, y_batch in train_loader:
		y_pred = model(X_batch)
		loss = criterion(y_pred, y_batch)
		total_loss += loss.item()
		loss.backward()
		optimizer.step()
		optimizer.zero_grad()
		metric.update(y_pred, y_batch)

	avg_loss = total_loss / len(train_loader)
	history['loss'].append(avg_loss)

	avg_metric_train = metric.compute().item()
	history['train_metric'].append(avg_metric_train)

history

{'loss': [1.3250573866951894,
  1.2384911059676451,
  1.094882480779627,
  0.830070633040972,
  0.6439116366884925,
  0.5939568769258573,
  0.572019193372779,
  0.5520172222586702,
  0.5386583653609615,
  0.5241985559545601],
 'train_metric': [0.9048439264297485,
  0.8787753582000732,
  0.8255719542503357,
  0.7128451466560364,
  0.6120055913925171,
  0.5755521059036255,
  0.5609515309333801,
  0.5494101047515869,
  0.5401222705841064,
  0.5313557982444763]}